# Day 27：Memory记忆机制：多轮会话隔离、窗口摘要压缩与SQLite持久化（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 4 周**：第4周：LangChain 核心架构与 RAG 文档预处理  
> **GitLab 工单**：`STARMIND-027` ｜ **开发分支**：`feat/day27-memory-sqlite-session`  
> **配套 HTML5 课件**：`courseware/day27/index.html` ｜ **独立版本快照**：`versions/day27/`

---

## 🎙️ 老陈开场旁白

早啊！今天聊聊让大模型「长记性」的核心技术——Memory 记忆机制。很多人以为多轮对话很简单，不就是拿个 Python `list` 把聊天记录存下来，每次全塞给大模型吗？真上生产这么写，三天必出大事故：第一，所有员工共用一个全局 list，张三的薪资咨询串到李四窗口里，直接 P0 级数据泄露；第二，聊到第 30 轮，历史消息飙到 20K Token，不仅账单爆炸，还会触发模型上下文超限报错；第三，服务一重启，所有聊天记录全丢。今天老陈带大家按 LangChain 0.3 最新规范，亲手实现 `SQLiteChatMessageHistory`（按 `session_id` 严格物理隔离）、`WindowMemory`、`SummaryBufferMemory` 和 `RunnableWithMessageHistory`！

### 承上启下
- **昨日基线**：昨天（Day 26）我们用 LCEL 管道符串起了多步推理流水线。但测试同学一测就发现：第一句说「我叫小王，在研发部」，第二句问「我叫什么名字？」，链直接装傻——因为大模型本身是无状态的（Stateless），每次 HTTP 请求都是全新失忆。
- **明日预告**：明天（Day 28）我们将正式迈出企业级 RAG 系统的第一步：多格式文档加载（PDF/Word/MD/HTML/CSV）与中文递归分块（RecursiveCharacterTextSplitter）。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day27'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-027）

星辰智脑内部助手即将在全公司 800 名员工中灰度推广，员工在查询报销、休假、IT 故障时往往需要连续追问 5~15 轮，且服务需支持重启后恢复历史会话。

**今日迭代目标：**
- 实现 `BaseChatMessageHistory` 抽象及基于 SQLite 的 `SQLiteChatMessageHistory`，严格按 `session_id` 隔离并支持持久化。
- 实现两种企业级记忆压缩策略：滑动窗口记忆（保留最近 K 轮）与摘要缓冲记忆（超出阈值将早期对话压缩为摘要）。
- 实现 `RunnableWithMessageHistory` 包装器，无侵入地为任意 LCEL 链注入多轮记忆能力。

**验收标准：**
- 不同 `session_id` 的并发读写完全隔离，绝不串台；重新实例化对象后可从 SQLite 恢复完整历史。
- 滑动窗口能精确截断保留最近 `k` 轮（`2*k` 条消息）；摘要记忆在超限时自动生成摘要并缩减消息数。
- 所有 pytest 单元测试（含跨会话隔离、重启持久化、窗口截断与摘要触发）100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么绝对不能用全局变量存聊天记录？

FastAPI/Uvicorn 在处理多用户并发请求时，模块级全局变量是被所有请求共享的。如果把 `memory = ConversationBufferMemory()` 定义在路由函数外面，A 员工刚输入完自己的手机号，B 员工下一秒就能问出来！正确做法是：**Chain 本身必须完全无状态**，每次请求通过 `session_id`（建议格式为 `{tenant_id}:{user_id}:{conv_id}`）从外部数据库动态拉取该用户的专属历史。

> **⚠️ 老陈避坑**：踩坑提醒：前端传过来的 `session_id` 绝对不能直接信任！生产环境必须结合 JWT Token 里的 `user_id` 校验该会话的归属权，防止越权窥探他人聊天记录（IDOR 漏洞）。

### 2. 2. 生产环境 LangChain 真实写法对照

在安装了 `langchain-community` 和 `langchain-core` 的项目中，核心写法与我们今日手写代码几乎一字不差：
```python
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(session_id=session_id, connection='sqlite:///starmind_chat.db')

with_memory = RunnableWithMessageHistory(
    runnable=prompt | llm,
    get_session_history=get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)
resp = with_memory.invoke({'input': '我刚才问了啥？'}, config={'configurable': {'session_id': 'u_1001'}})
```

> **⚠️ 老陈避坑**：踩坑提醒：使用 `trim_messages` 或窗口裁剪时，必须保证截断后的第一条历史消息不是孤立的 `AIMessage`（某些大模型 API 如 Anthropic/部分 Qwen 模板严格要求 `user -> assistant` 交替出现），因此按「整轮（2 条消息为一组）」裁剪是最稳妥的工程实践。

### 3. 3. SQLite vs Redis 怎么选？

中小团队 MVP 或单机内网部署（日活 < 1000）直接用 SQLite WAL 模式足够稳定，还能免运维直接做 SQL 审计；一旦进入多副本 K8s 水平扩展，必须把近期热会话放 Redis（设置 24 小时 TTL），全量冷数据异步归档到 PostgreSQL 或 MySQL。

> **⚠️ 老陈避坑**：踩坑提醒：SQLite 默认在多线程下会报 `SQLite objects created in a thread can only be used in that same thread`，生产封装时要么每次操作短连接 + `threading.Lock`，要么开启连接池与 WAL 模式。


## 💻 实操步骤 1：步骤一：实现线程安全、按 session_id 隔离的 SQLiteChatMessageHistory

- **目标文件**：`versions/day27/src/day27_sqlite_history.py`
- **老陈代码旁白**：咱们先用 Python 标准库 `sqlite3` 撸一个生产级会话历史存储器。重点关注三点：表结构与复合索引设计、按 `session_id` 严格隔离、以及支持序列化还原为 `HumanMessage` / `AIMessage` / `SystemMessage`。


In [ ]:
"""
Day 27 - 步骤一：基于 SQLite 的多会话隔离历史存储（SQLiteChatMessageHistory）
对应生产包：langchain_community.chat_message_histories.SQLChatMessageHistory
"""

from __future__ import annotations

import json
import sqlite3
import sys
import threading
import time
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict, List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day25_chat_model import AIMessage, BaseMessage, HumanMessage, SystemMessage


class BaseChatMessageHistory(ABC):
    """对话历史存储抽象基类。"""

    @property
    @abstractmethod
    def messages(self) -> List[BaseMessage]:
        """返回当前会话按时间升序排列的所有消息。"""

    @abstractmethod
    def add_message(self, message: BaseMessage) -> None:
        """追加单条消息。"""

    def add_user_message(self, content: str) -> None:
        self.add_message(HumanMessage(content=content))

    def add_ai_message(self, content: str) -> None:
        self.add_message(AIMessage(content=content))

    @abstractmethod
    def clear(self) -> None:
        """清空当前会话的全部消息（不影响其他 session_id）。"""


class InMemoryChatMessageHistory(BaseChatMessageHistory):
    """内存态会话存储，适合临时脚本与轻量单测。"""

    def __init__(self) -> None:
        self._messages: List[BaseMessage] = []

    @property
    def messages(self) -> List[BaseMessage]:
        return list(self._messages)

    def add_message(self, message: BaseMessage) -> None:
        self._messages.append(message)

    def clear(self) -> None:
        self._messages.clear()


class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """
    企业级 SQLite 持久化会话历史：
    - 通过 session_id 严格隔离多员工对话。
    - 内置复合索引 (session_id, id) 与线程锁，支持多线程并发读写。
    """

    _lock = threading.Lock()
    _ROLE_TO_CLS = {
        "system": SystemMessage,
        "user": HumanMessage,
        "assistant": AIMessage,
    }

    def __init__(self, session_id: str, db_path: str = "starmind_memory.db") -> None:
        if not session_id or not session_id.strip():
            raise ValueError("session_id 不能为空")
        self.session_id = session_id.strip()
        self.db_path = db_path
        self._init_db()

    def _connect(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path, timeout=10.0)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_db(self) -> None:
        with self._lock, self._connect() as conn:
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS chat_messages (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL,
                    content TEXT NOT NULL,
                    metadata_json TEXT NOT NULL DEFAULT '{}',
                    created_at REAL NOT NULL
                )
                """
            )
            conn.execute(
                """
                CREATE INDEX IF NOT EXISTS idx_chat_session_id
                ON chat_messages (session_id, id)
                """
            )

    @property
    def messages(self) -> List[BaseMessage]:
        with self._lock, self._connect() as conn:
            rows = conn.execute(
                "SELECT role, content, metadata_json FROM chat_messages WHERE session_id = ? ORDER BY id ASC",
                (self.session_id,),
            ).fetchall()
        result: List[BaseMessage] = []
        for row in rows:
            cls = self._ROLE_TO_CLS.get(row["role"], HumanMessage)
            meta = json.loads(row["metadata_json"])
            msg = cls(content=row["content"])
            msg.additional_kwargs = meta
            result.append(msg)
        return result

    def add_message(self, message: BaseMessage) -> None:
        with self._lock, self._connect() as conn:
            conn.execute(
                """
                INSERT INTO chat_messages (session_id, role, content, metadata_json, created_at)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    self.session_id,
                    message.role,
                    message.content,
                    json.dumps(message.additional_kwargs, ensure_ascii=False),
                    time.time(),
                ),
            )

    def clear(self) -> None:
        with self._lock, self._connect() as conn:
            conn.execute("DELETE FROM chat_messages WHERE session_id = ?", (self.session_id,))

    @classmethod
    def list_sessions(cls, db_path: str) -> List[str]:
        """运维辅助：列出数据库中所有活跃的 session_id。"""
        with cls._lock, sqlite3.connect(db_path) as conn:
            rows = conn.execute(
                "SELECT DISTINCT session_id FROM chat_messages ORDER BY session_id ASC"
            ).fetchall()
        return [r[0] for r in rows]


if __name__ == "__main__":
    import tempfile

    with tempfile.TemporaryDirectory() as tmpdir:
        db = str(Path(tmpdir) / "demo_chat.db")
        h_zhang = SQLiteChatMessageHistory("emp_zhangsan", db_path=db)
        h_li = SQLiteChatMessageHistory("emp_lisi", db_path=db)

        h_zhang.add_user_message("我叫张三，在杭州星辰数科财务部")
        h_zhang.add_ai_message("你好张三，请问财务报销有什么可以帮您？")
        h_li.add_user_message("我叫李四，VPN连不上了")

        print("活跃会话:", SQLiteChatMessageHistory.list_sessions(db))
        print("张三历史条数:", len(h_zhang.messages), "| 内容:", [m.content for m in h_zhang.messages])
        print("李四历史条数:", len(h_li.messages), "| 内容:", [m.content for m in h_li.messages])


### 🔍 步骤 1 代码深度解析

1. 使用 `CREATE INDEX IF NOT EXISTS idx_chat_session_id ON chat_messages (session_id, id)`，这样即使表里存了 100 万条员工消息，查某个 `session_id` 的历史也是毫秒级索引扫描。
2. 用 `threading.Lock()` + `with self._connect() as conn:` 既保证了多线程写入不报 `database is locked`，又确保每次写完自动提交事务。
3. `clear()` 严格带 `WHERE session_id = ?`，只删当前会话，绝不误删其他同事记录。

**预期运行输出参考：**
```text
活跃会话: ['emp_lisi', 'emp_zhangsan']
张三历史条数: 2 | 内容: ['我叫张三，在杭州星辰数科财务部', '你好张三，请问财务报销有什么可以帮您？']
李四历史条数: 1 | 内容: ['我叫李四，VPN连不上了']
```


## 💻 实操步骤 2：步骤二：实现窗口记忆、摘要缓冲记忆与 RunnableWithMessageHistory

- **目标文件**：`versions/day27/src/day27_memory_chain.py`
- **老陈代码旁白**：存储层稳了，现在把「滑动窗口裁剪」、「摘要压缩」和 `RunnableWithMessageHistory` 装配起来，让任何无状态的 LCEL 链秒变支持多轮记忆、绝不失忆也不爆 Token 的智能对话引擎！


In [ ]:
"""
Day 27 - 步骤二：窗口记忆、摘要缓冲记忆与 RunnableWithMessageHistory 包装器
对应生产包：langchain_core.runnables.history.RunnableWithMessageHistory
"""

from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day25_chat_model import AIMessage, BaseMessage, HumanMessage, StarMindChatModel, SystemMessage
from src.day26_lcel_core import Runnable
from src.day27_sqlite_history import BaseChatMessageHistory, SQLiteChatMessageHistory


class WindowMemoryStrategy:
    """滑动窗口记忆策略：仅保留最近 k 轮对话（每轮 = 1条用户 + 1条AI）。"""

    def __init__(self, k: int = 3) -> None:
        if k < 1:
            raise ValueError("窗口轮数 k 必须 >= 1")
        self.k = k

    def compress(self, messages: List[BaseMessage]) -> List[BaseMessage]:
        max_msgs = self.k * 2
        if len(messages) <= max_msgs:
            return list(messages)
        return list(messages[-max_msgs:])


class SummaryBufferMemoryStrategy:
    """
    摘要缓冲记忆策略（对应 ConversationSummaryBufferMemory）：
    - 当历史消息总字符数 <= max_chars 时原样保留；
    - 超限时将早期消息压缩为一条 SystemMessage 摘要，并保留最近 keep_recent_turns 轮原文。
    """

    def __init__(
        self,
        max_chars: int = 180,
        keep_recent_turns: int = 2,
        summarizer: Optional[Callable[[List[BaseMessage]], str]] = None,
    ) -> None:
        self.max_chars = max_chars
        self.keep_recent_turns = keep_recent_turns
        self._summarizer = summarizer or self._default_extractive_summarizer

    @staticmethod
    def _default_extractive_summarizer(older_msgs: List[BaseMessage]) -> str:
        """离线确定性摘要器（生产环境可替换为轻量 LLM 摘要链）。"""
        facts: List[str] = []
        for msg in older_msgs:
            if msg.role == "user":
                snippet = msg.content.strip().replace("\n", " ")[:40]
                facts.append(f"用户提及「{snippet}」")
        joined = "；".join(facts) if facts else "此前进行了常规寒暄"
        return f"【早期对话摘要】{joined}。"

    def compress(self, messages: List[BaseMessage]) -> List[BaseMessage]:
        total_chars = sum(len(m.content) for m in messages)
        keep_count = self.keep_recent_turns * 2
        if total_chars <= self.max_chars or len(messages) <= keep_count:
            return list(messages)
        older = messages[:-keep_count]
        recent = messages[-keep_count:]
        summary_text = self._summarizer(older)
        return [SystemMessage(content=summary_text), *recent]


class RunnableWithMessageHistory(Runnable[Dict[str, Any], AIMessage]):
    """
    无侵入多轮记忆包装器：
    自动根据 session_id 读取历史 -> 执行记忆裁剪/摘要 -> 注入 history 运行底层链 -> 写回本轮对话。
    """

    def __init__(
        self,
        runnable: Runnable[Dict[str, Any], AIMessage],
        get_session_history: Callable[[str], BaseChatMessageHistory],
        memory_strategy: WindowMemoryStrategy | SummaryBufferMemoryStrategy | None = None,
        input_messages_key: str = "input",
        history_messages_key: str = "history",
    ) -> None:
        self.runnable = runnable
        self.get_session_history = get_session_history
        self.memory_strategy = memory_strategy or WindowMemoryStrategy(k=5)
        self.input_messages_key = input_messages_key
        self.history_messages_key = history_messages_key

    def invoke(self, input_data: Dict[str, Any], config: Optional[Dict[str, Any]] = None) -> AIMessage:
        session_id = (config or {}).get("configurable", {}).get("session_id")
        if not session_id:
            raise ValueError("调用 RunnableWithMessageHistory 必须在 config.configurable 中提供 session_id")

        history_store = self.get_session_history(session_id)
        compressed_history = self.memory_strategy.compress(history_store.messages)

        user_input = str(input_data[self.input_messages_key])
        payload = {**input_data, self.history_messages_key: compressed_history}
        ai_msg = self.runnable.invoke(payload)

        # 对话完成后再持久化落盘（确保异常时不污染历史）
        history_store.add_user_message(user_input)
        history_store.add_ai_message(ai_msg.content)
        return ai_msg


def _memory_aware_chat_handler(messages: List[BaseMessage]) -> str:
    """模拟能读取多轮历史上下文（含摘要）的星辰智能助手。"""
    full_context = "\n".join(m.content for m in messages[:-1])
    question = messages[-1].content

    # 简单从历史上下文中提取员工姓名与部门，验证记忆是否生效
    name_match = re.search(r"我叫([\u4e00-\u9fffA-Za-z0-9]{2,8})", full_context)
    dept_match = re.search(r"在([\u4e00-\u9fffA-Za-z0-9]{2,10}部)", full_context)

    if "我是谁" in question or "我叫什么" in question:
        if name_match:
            dept_info = f"，来自{dept_match.group(1)}" if dept_match else ""
            return f"我记得您！您是{name_match.group(1)}{dept_info}。"
        return "抱歉，当前会话记忆里还没有您的姓名信息，方便告诉我怎么称呼您吗？"

    return f"已收到您的消息：「{question}」（当前参考历史消息 {len(messages) - 1} 条）。"


class ContextualChatChain(Runnable[Dict[str, Any], AIMessage]):
    """将 system + history + current_input 拼装后送入 ChatModel。"""

    def __init__(self, system_prompt: str) -> None:
        self.system_prompt = system_prompt
        self.model = StarMindChatModel(model_name="starmind-memory-llm", handler=_memory_aware_chat_handler)

    def invoke(self, input_data: Dict[str, Any]) -> AIMessage:
        msgs: List[BaseMessage] = [SystemMessage(content=self.system_prompt)]
        msgs.extend(input_data.get("history", []))
        msgs.append(HumanMessage(content=input_data["input"]))
        return self.model.invoke(msgs)


if __name__ == "__main__":
    import tempfile

    with tempfile.TemporaryDirectory() as tmpdir:
        db_file = str(Path(tmpdir) / "starmind_sessions.db")
        base_chain = ContextualChatChain("你是杭州星辰数科企业助手，请结合上下文准确回答。")
        chat_with_memory = RunnableWithMessageHistory(
            runnable=base_chain,
            get_session_history=lambda sid: SQLiteChatMessageHistory(sid, db_path=db_file),
            memory_strategy=SummaryBufferMemoryStrategy(max_chars=120, keep_recent_turns=1),
        )

        cfg_a = {"configurable": {"session_id": "emp_wang"}}
        cfg_b = {"configurable": {"session_id": "emp_chen"}}

        print("小王 第1轮:", chat_with_memory.invoke({"input": "你好，我叫王小明，在研发部做Python后端开发"}, config=cfg_a).content)
        print("小王 第2轮:", chat_with_memory.invoke({"input": "请问今年年假怎么折算，另外出差补贴标准是多少？"}, config=cfg_a).content)
        print("小王 第3轮:", chat_with_memory.invoke({"input": "考考你，我叫什么名字？"}, config=cfg_a).content)
        print("老陈 第1轮:", chat_with_memory.invoke({"input": "考考你，我叫什么名字？"}, config=cfg_b).content)


### 🔍 步骤 2 代码深度解析

1. 注意看第 3 轮时，由于前两轮对话字符数超过了 `max_chars=120`，`SummaryBufferMemoryStrategy` 自动把第 1 轮压缩进了 `【早期对话摘要】`，所以即使滑动窗口只保留最近 1 轮，小王问「我叫什么名字」依然能精准答出「王小明，来自研发部」！
2. 同时，老陈使用 `session_id='emp_chen'` 提问时，系统回答「没有您的姓名信息」，证明了多会话绝对隔离。
3. 落盘动作放在 `self.runnable.invoke()` 成功返回之后，避免大模型报错时把残缺的半轮对话写进数据库。

**预期运行输出参考：**
```text
小王 第1轮: 已收到您的消息：「你好，我叫王小明，在研发部做Python后端开发」（当前参考历史消息 1 条）。
小王 第2轮: 已收到您的消息：「请问今年年假怎么折算，另外出差补贴标准是多少？」（当前参考历史消息 3 条）。
小王 第3轮: 我记得您！您是王小明，来自研发部。
老陈 第1轮: 抱歉，当前会话记忆里还没有您的姓名信息，方便告诉我怎么称呼您吗？
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day27/tests/test_day27.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 27 自动化单元测试：验证 SQLite 持久化、多会话隔离、窗口裁剪与摘要压缩。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day25_chat_model import HumanMessage, AIMessage, SystemMessage
from src.day27_sqlite_history import SQLiteChatMessageHistory
from src.day27_memory_chain import (
    ContextualChatChain,
    RunnableWithMessageHistory,
    SummaryBufferMemoryStrategy,
    WindowMemoryStrategy,
)


def test_sqlite_persistence_and_isolation(tmp_path):
    db_path = str(tmp_path / "test_mem.db")
    h1 = SQLiteChatMessageHistory("sess_a", db_path=db_path)
    h2 = SQLiteChatMessageHistory("sess_b", db_path=db_path)
    h1.add_user_message("A的消息")
    h2.add_user_message("B的消息")

    # 模拟服务重启：重新实例化读取
    h1_reloaded = SQLiteChatMessageHistory("sess_a", db_path=db_path)
    assert len(h1_reloaded.messages) == 1
    assert h1_reloaded.messages[0].content == "A的消息"

    h1_reloaded.clear()
    assert len(h1_reloaded.messages) == 0
    assert len(SQLiteChatMessageHistory("sess_b", db_path=db_path).messages) == 1


def test_window_and_summary_strategies():
    msgs = []
    for i in range(5):
        msgs.append(HumanMessage(content=f"第{i}轮用户提问：非常详细的业务背景描述内容"))
        msgs.append(AIMessage(content=f"第{i}轮AI回复：好的收到"))

    win = WindowMemoryStrategy(k=2)
    trimmed = win.compress(msgs)
    assert len(trimmed) == 4
    assert "第3轮" in trimmed[0].content

    summary = SummaryBufferMemoryStrategy(max_chars=80, keep_recent_turns=1)
    compressed = summary.compress(msgs)
    assert isinstance(compressed[0], SystemMessage)
    assert "早期对话摘要" in compressed[0].content
    assert len(compressed) == 3  # 1条摘要 + 最近1轮(2条)


def test_runnable_with_message_history_end_to_end(tmp_path):
    db_path = str(tmp_path / "e2e.db")
    chain = RunnableWithMessageHistory(
        runnable=ContextualChatChain("你是星辰助手"),
        get_session_history=lambda sid: SQLiteChatMessageHistory(sid, db_path=db_path),
        memory_strategy=SummaryBufferMemoryStrategy(max_chars=100, keep_recent_turns=1),
    )

    cfg_u1 = {"configurable": {"session_id": "u1"}}
    cfg_u2 = {"configurable": {"session_id": "u2"}}

    chain.invoke({"input": "你好，我叫林悦，在产品部"}, config=cfg_u1)
    chain.invoke({"input": "请帮我查一下下周会议室预定规则和投影仪报修流程"}, config=cfg_u1)
    r1 = chain.invoke({"input": "我叫什么？"}, config=cfg_u1)
    r2 = chain.invoke({"input": "我叫什么？"}, config=cfg_u2)

    assert "林悦" in r1.content and "产品部" in r1.content
    assert "还没有您的姓名" in r2.content

    with pytest.raises(ValueError, match="session_id"):
        chain.invoke({"input": "没有传session_id"})

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现按 Token 预算精确裁剪的 TokenBudgetMemoryStrategy

**题目要求：**
基于 Day 25 的 Token 估算思想，实现 `TokenBudgetMemoryStrategy(max_tokens)`：从最新一轮对话开始往前累加，尽可能保留更多完整对话轮次，直到总 Token 数即将超过 `max_tokens` 时停止（至少保留最新 1 轮，且按偶数组成对保留）。

**思路提示：** 逆序每次取一对 `(human, ai)` 消息，计算 Token 累加，超限即 `break`，最后再反转回正序。


In [ ]:
"""Day 27 作业一参考答案：按 Token 预算精确裁剪记忆"""
import sys
from pathlib import Path
from typing import List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day25_chat_model import AIMessage, BaseChatModel, BaseMessage, HumanMessage


class TokenBudgetMemoryStrategy:
    def __init__(self, max_tokens: int = 120) -> None:
        self.max_tokens = max_tokens

    def compress(self, messages: List[BaseMessage]) -> List[BaseMessage]:
        if len(messages) <= 2:
            return list(messages)
        selected_rev: List[BaseMessage] = []
        used = 0
        # 按 (human, ai) 成对从后往前扫描
        i = len(messages) - 1
        while i >= 1:
            pair = [messages[i - 1], messages[i]]
            pair_tokens = sum(BaseChatModel._estimate_tokens(m.content) for m in pair)
            if selected_rev and used + pair_tokens > self.max_tokens:
                break
            selected_rev.extend(reversed(pair))
            used += pair_tokens
            i -= 2
        return list(reversed(selected_rev))


if __name__ == "__main__":
    msgs = []
    for idx in range(6):
        msgs.append(HumanMessage(content=f"问题{idx}：请详细介绍杭州星辰数科的差旅住宿报销标准与审批流"))
        msgs.append(AIMessage(content=f"回答{idx}：一线城市每日上限600元，需部门主管审批"))
    strategy = TokenBudgetMemoryStrategy(max_tokens=60)
    kept = strategy.compress(msgs)
    print("原消息数:", len(msgs), "-> 裁剪后消息数:", len(kept))
    assert len(kept) < len(msgs) and len(kept) % 2 == 0


**答案解析与评分标准：**

相比固定保留 K 轮，按 Token 预算裁剪能精确防止某轮用户粘贴 5000 字长日志导致上下文溢出，是生产环境最常用的策略。


### 作业 2（企业实战）：作业二：给 SQLiteChatMessageHistory 增加会话 TTL 自动过期清理

**题目要求：**
为 `SQLiteChatMessageHistory` 编写一个辅助类或方法 `purge_expired_sessions(db_path, ttl_seconds)`：删除所有 `created_at < now - ttl_seconds` 的过期消息记录，并返回清理掉的行数。

**思路提示：** 用 `DELETE FROM chat_messages WHERE created_at < ?`，读取 `cursor.rowcount` 返回删除行数。


In [ ]:
"""Day 27 作业二参考答案：SQLite 会话历史 TTL 过期清理"""
import sqlite3
import sys
import tempfile
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day27_sqlite_history import SQLiteChatMessageHistory


def purge_expired_messages(db_path: str, ttl_seconds: float, now: float | None = None) -> int:
    cutoff = (now or time.time()) - ttl_seconds
    with sqlite3.connect(db_path) as conn:
        cur = conn.execute("DELETE FROM chat_messages WHERE created_at < ?", (cutoff,))
        return cur.rowcount


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmpdir:
        db = str(Path(tmpdir) / "ttl.db")
        h = SQLiteChatMessageHistory("emp_01", db_path=db)
        h.add_user_message("一条旧消息")
        # 模拟 2 天后清理 TTL=86400 的过期消息
        deleted = purge_expired_messages(db, ttl_seconds=86400, now=time.time() + 172800)
        print("清理过期消息条数:", deleted)
        assert deleted == 1
        assert len(h.messages) == 0


**答案解析与评分标准：**

企业合规（如《个人信息保护法》）通常要求对话日志不得无限期留存。定期清理过期记录既控磁盘成本又满足审计要求。


### 作业 3（架构进阶）：作业三：实现带用户越权防护的 SafeSessionFactory

**题目要求：**
实现 `SafeSessionFactory(db_path)`：要求调用 `get_history(user_id, conv_id)` 时强制拼装复合主键 `f'{user_id}::{conv_id}'`，并禁止 `user_id` 或 `conv_id` 中包含 `::` 或空白字符，彻底杜绝水平越权（IDOR）。

**思路提示：** 在工厂方法里先校验合法性，再返回 `SQLiteChatMessageHistory(session_id=f'{user_id}::{conv_id}', db_path=...)`。


In [ ]:
"""Day 27 作业三参考答案：防水平越权（IDOR）的安全会话工厂"""
import re
import sys
import tempfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day27_sqlite_history import SQLiteChatMessageHistory


class SafeSessionFactory:
    VALID_ID_RE = re.compile(r"^[A-Za-z0-9_-]{1,64}$")

    def __init__(self, db_path: str) -> None:
        self.db_path = db_path

    def get_history(self, user_id: str, conv_id: str) -> SQLiteChatMessageHistory:
        if not self.VALID_ID_RE.match(user_id) or not self.VALID_ID_RE.match(conv_id):
            raise ValueError("非法的 user_id 或 conv_id，存在注入或越权风险")
        scoped_session_id = f"{user_id}::{conv_id}"
        return SQLiteChatMessageHistory(session_id=scoped_session_id, db_path=self.db_path)


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmpdir:
        factory = SafeSessionFactory(str(Path(tmpdir) / "safe.db"))
        h_a = factory.get_history("user_1001", "conv_1")
        h_b = factory.get_history("user_1002", "conv_1")
        h_a.add_user_message("仅1001可见的机密对话")
        assert len(h_b.messages) == 0
        print("越权隔离校验通过，user_1002 无法访问 conv_1 的 1001 数据。")


**答案解析与评分标准：**

即使两个员工的会话 ID 都叫 `conv_1`，通过服务端从鉴权 Token 中强制注入 `user_id::` 前缀，从根源上消灭水平越权漏洞。


---

## 🎯 老陈收尾复盘

今天大家亲手踩实了多轮对话的三个关键工程点：持久化（SQLite 落盘不丢）、隔离性（`session_id` 绝不串台）、以及有界上下文（滑动窗口 + 增量摘要压缩）。记住了，生产环境永远不要无上限拼接历史消息！明天咱们正式切入 RAG 文档加载与中文切分。
